In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH, "data", "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH, "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH, "reports", "tables"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH, "reports", "figures"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

delhi_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase2_audit.parquet"
    )
)

delhi_df["Datetime"] = pd.to_datetime(
    delhi_df["Datetime"],
    errors="coerce"
)

print("Phase 2 checkpoint restored.")
print("Rows:", len(delhi_df))
print("Stations:", delhi_df["StationId"].nunique())

In [ ]:
clean_df = delhi_df.copy()

print("Working copy created.")
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))

In [ ]:
clean_df = clean_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Data sorted by StationId and Datetime.")

In [ ]:
columns_to_drop = [
    "Xylene",
    "AQI_Bucket"
]

clean_df = clean_df.drop(
    columns=columns_to_drop
)

print("Removed columns:", columns_to_drop)
print("Remaining columns:", len(clean_df.columns))

In [ ]:
pollutant_columns = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

target_column = "AQI"

print("Pollutant features:")
print(pollutant_columns)

print("\nTarget:")
print(target_column)

In [ ]:
for column in pollutant_columns + [target_column]:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

print(clean_df[pollutant_columns + [target_column]].dtypes)

In [ ]:
negative_counts = (
    clean_df[pollutant_columns + [target_column]]
    .lt(0)
    .sum()
)

print(negative_counts)

In [ ]:
for column in pollutant_columns:
    clean_df[f"{column}_was_missing"] = (
        clean_df[column].isna().astype("int8")
    )

print("Missingness indicators created.")

In [ ]:
MAX_FFILL_HOURS = 3

clean_df[pollutant_columns] = (
    clean_df
    .groupby("StationId")[pollutant_columns]
    .ffill(limit=MAX_FFILL_HOURS)
)

print(
    f"Short gaps filled using past values only, "
    f"with a maximum limit of {MAX_FFILL_HOURS} hours."
)

In [ ]:
remaining_missing = (
    clean_df[pollutant_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .to_frame("RemainingMissingPercentage")
)

display(remaining_missing)

In [ ]:
print(
    "Remaining missing predictor values will NOT be "
    "filled using future observations."
)

print(
    "They will be handled later inside the model-training "
    "pipeline using training data only."
)

In [ ]:
before_aqi_drop = len(clean_df)

clean_df = clean_df.dropna(
    subset=["AQI"]
).copy()

after_aqi_drop = len(clean_df)

print("Rows before:", before_aqi_drop)
print("Rows after: ", after_aqi_drop)
print("Rows removed:", before_aqi_drop - after_aqi_drop)

In [ ]:
clean_df["AQI_t_plus_1"] = (
    clean_df
    .groupby("StationId")["AQI"]
    .shift(-1)
)

print("Future target created.")

In [ ]:
verification = (
    clean_df[
        [
            "StationId",
            "StationName",
            "Datetime",
            "AQI",
            "AQI_t_plus_1"
        ]
    ]
    .sort_values(
        ["StationId", "Datetime"]
    )
)

display(verification.head(10))

In [ ]:
before_target_drop = len(clean_df)

clean_df = clean_df.dropna(
    subset=["AQI_t_plus_1"]
).copy()

after_target_drop = len(clean_df)

print("Rows before:", before_target_drop)
print("Rows after: ", after_target_drop)
print("Rows removed:", before_target_drop - after_target_drop)

In [ ]:
print(
    "Missing future AQI:",
    clean_df["AQI_t_plus_1"].isna().sum()
)

print(
    "Negative future AQI:",
    (clean_df["AQI_t_plus_1"] < 0).sum()
)

In [ ]:
clean_df["Hour"] = clean_df["Datetime"].dt.hour
clean_df["Day"] = clean_df["Datetime"].dt.day
clean_df["Month"] = clean_df["Datetime"].dt.month
clean_df["Year"] = clean_df["Datetime"].dt.year
clean_df["DayOfWeek"] = clean_df["Datetime"].dt.dayofweek

clean_df["IsWeekend"] = (
    clean_df["DayOfWeek"] >= 5
).astype("int8")

print(
    clean_df[
        [
            "Datetime",
            "Hour",
            "Day",
            "Month",
            "Year",
            "DayOfWeek",
            "IsWeekend"
        ]
    ].head()
)

In [ ]:
def assign_season(month):
    if month in [10, 11]:
        return "PostMonsoon"
    elif month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Other"

clean_df["Season"] = (
    clean_df["Month"]
    .apply(assign_season)
)

print(
    clean_df["Season"].value_counts()
)

In [ ]:
clean_df["IsStubbleSeason"] = (
    clean_df["Month"]
    .isin([10, 11])
).astype("int8")

print(
    clean_df["IsStubbleSeason"].value_counts()
)

In [ ]:
clean_df["IsDiwaliWindow"] = 0

print(
    "Diwali flag initialized; exact annual windows "
    "will be defined in Phase 5."
)

In [ ]:
aqi_q01 = clean_df["AQI"].quantile(0.01)
aqi_q99 = clean_df["AQI"].quantile(0.99)

clean_df["AQI_extreme_flag"] = (
    (clean_df["AQI"] <= aqi_q01) |
    (clean_df["AQI"] >= aqi_q99)
).astype("int8")

print("AQI 1st percentile:", aqi_q01)
print("AQI 99th percentile:", aqi_q99)
print(
    "Extreme AQI observations:",
    clean_df["AQI_extreme_flag"].sum()
)

In [ ]:
print("Cleaned dataset shape:", clean_df.shape)
print("\nColumns:")
print(clean_df.columns.tolist())

In [ ]:
remaining_missing_all = (
    clean_df
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(
    remaining_missing_all[
        remaining_missing_all > 0
    ].to_frame("MissingPercentage")
)

In [ ]:
clean_dataset_path = os.path.join(
    PROCESSED_PATH,
    "delhi_clean.parquet"
)

clean_df.to_parquet(
    clean_dataset_path,
    index=False
)

print("Clean dataset saved:")
print(clean_dataset_path)

In [ ]:
cleaning_summary = {
    "input_rows": int(len(delhi_df)),
    "output_rows": int(len(clean_df)),
    "rows_removed_for_missing_current_aqi": int(
        before_aqi_drop - after_aqi_drop
    ),
    "rows_removed_for_missing_future_aqi": int(
        before_target_drop - after_target_drop
    ),
    "dropped_columns": [
        "Xylene",
        "AQI_Bucket"
    ],
    "forward_fill_limit_hours": MAX_FFILL_HOURS,
    "target": "AQI_t_plus_1",
    "negative_values_found": int(
        negative_counts.sum()
    )
}

summary_df = pd.DataFrame(
    [cleaning_summary]
)

display(summary_df)

In [ ]:
cleaning_summary_path = os.path.join(
    TABLE_PATH,
    "phase3_cleaning_summary.csv"
)

summary_df.to_csv(
    cleaning_summary_path,
    index=False
)

print("Cleaning summary saved:")
print(cleaning_summary_path)

In [ ]:
print("=" * 60)
print("PHASE 3 — CLEANING & PREPROCESSING SUMMARY")
print("=" * 60)

print("Rows:", len(clean_df))
print("Stations:", clean_df["StationId"].nunique())
print("Start:", clean_df["Datetime"].min())
print("End:", clean_df["Datetime"].max())

print(
    "Missing target:",
    clean_df["AQI_t_plus_1"].isna().sum()
)

print(
    "Negative target:",
    (clean_df["AQI_t_plus_1"] < 0).sum()
)

print(
    "Xylene present:",
    "Xylene" in clean_df.columns
)

print(
    "AQI_Bucket present:",
    "AQI_Bucket" in clean_df.columns
)

print("=" * 60)

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save final Phase 3 dataset
# --------------------------------------------------

clean_checkpoint = os.path.join(
    PROCESSED_PATH,
    "delhi_phase3_clean.parquet"
)

clean_df.to_parquet(
    clean_checkpoint,
    index=False
)

# --------------------------------------------------
# 2. Save cleaning configuration
# --------------------------------------------------

cleaning_config = {
    "dropped_columns": [
        "Xylene",
        "AQI_Bucket"
    ],
    "target": "AQI_t_plus_1",
    "forward_fill_limit_hours": 3,
    "preserve_extreme_aqi": True,
    "event_flags_created": [
        "IsWeekend",
        "IsStubbleSeason",
        "IsDiwaliWindow"
    ],
    "season_feature": "Season"
}

config_path = os.path.join(
    OUTPUT_PATH,
    "phase3_cleaning_config.json"
)

with open(config_path, "w") as f:
    json.dump(
        cleaning_config,
        f,
        indent=4
    )

# --------------------------------------------------
# 3. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 3 - Data Cleaning & Preprocessing",
    "next_phase": "Phase 4 - Exploratory Data Analysis",
    "rows": int(len(clean_df)),
    "stations": int(clean_df["StationId"].nunique()),
    "start_datetime": str(clean_df["Datetime"].min()),
    "end_datetime": str(clean_df["Datetime"].max()),
    "target": "AQI_t_plus_1"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(status_path, "w") as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 4. Verification
# --------------------------------------------------

files_to_check = [
    clean_checkpoint,
    config_path,
    status_path
]

print("=" * 60)
print("PHASE 3 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 3")
print("Next: Phase 4 - Exploratory Data Analysis")